# TokenGate: find a needle, read fewer tokens

[![GitHub](https://img.shields.io/badge/GitHub-TokenGate-181717?logo=github&logoColor=white)](https://github.com/token-gate-llms/TokenGate) [![arXiv](https://img.shields.io/badge/arXiv-2610.26754-B31B1B?logo=arxiv&logoColor=white)](https://arxiv.org/abs/2610.26754) [![X](https://img.shields.io/badge/X-Launch%20thread-000000?logo=x&logoColor=white)](https://x.com/lianapatel_/status/2112924355240099875) [![HF Qwen3](https://img.shields.io/badge/%F0%9F%A4%97-Qwen3--1.7B-FFD21E)](https://huggingface.co/liana313/tokengate-niah-qwen3-1.7b) [![HF Qwen3.5](https://img.shields.io/badge/%F0%9F%A4%97-Qwen3.5--2B-FFD21E)](https://huggingface.co/liana313/tokengate-niah-qwen3.5-2b)

TokenGate adds a learned gate near the start of an existing model. It scores the document, keeps the most useful **K tokens**, and lets later layers process those tokens. The question and instructions stay intact. This demo uses the released NIAH gate and its paired LoRA adapter.

![TokenGate overview](https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/fig1_overview.png)

Run the cells in order. Sections: **Setup and information → Inference → Optional training**. The default targets a free Colab T4; training needs an A100-class GPU.

## 1. Setup and information

**NIAH task:** one sentence is planted in a long document made from held-out essays. The model answers a question about that sentence. We use **three different city facts**, with the needle near the beginning, middle, and end of separate documents. At 16k, all three rows come from the frozen 150-row test partition. K is the maximum number of **document** tokens TokenGate keeps for later layers; `None` keeps all.

Qwen3.5-2B has a [released NIAH gate and paired LoRA](https://huggingface.co/liana313/tokengate-niah-qwen3.5-2b). On a free T4, choose up to 8k for single-pass inference; 16k needs the A100 option. Qwen3-1.7B remains the default.

In [ ]:
#@title Demo settings { display-mode: "form" }
MODEL = 'Qwen3-1.7B' #@param ['Qwen3-1.7B', 'Qwen3.5-2B']
DOC_TOKENS = '16k' #@param ['2k', '4k', '8k', '16k']
TOKEN_BUDGETS = '100, 500, 2000' #@param {type:"string"}
GPU_RUNTIME = 'T4' #@param ['T4', 'A100']

BASE_MODELS = {'Qwen3-1.7B':'Qwen/Qwen3-1.7B', 'Qwen3.5-2B':'Qwen/Qwen3.5-2B'}
CHECKPOINTS = {'Qwen3-1.7B':('liana313/tokengate-niah-qwen3-1.7b','3ef4d84ebd161a9700acced6e92f7cd5c50225ad'),
               'Qwen3.5-2B':('liana313/tokengate-niah-qwen3.5-2b','b1fe3e72b9896efc821fbfa02a19fcded4de6d77')}
BASE_MODEL = BASE_MODELS[MODEL]
DOC_TOKENS = int(DOC_TOKENS[:-1]) * 1024
assert MODEL != 'Qwen3.5-2B' or GPU_RUNTIME == 'A100' or DOC_TOKENS <= 8192, 'On a T4, Qwen3.5 supports up to 8k; choose an A100 for 16k.'
BUDGETS = tuple(int(x.strip()) for x in TOKEN_BUDGETS.split(',') if x.strip())
assert BUDGETS and len(set(BUDGETS)) == len(BUDGETS) and all(k > 0 for k in BUDGETS)
print(BASE_MODEL, '|', DOC_TOKENS, 'document tokens | K =', BUDGETS, '| GPU:', GPU_RUNTIME)

In [ ]:
#@title Install and import the API { display-mode: "form" }
import gc, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
assert GPU_RUNTIME in torch.cuda.get_device_name(0), f'Select a {GPU_RUNTIME} GPU in Colab: Runtime → Change runtime type.'
SOURCE = Path.cwd() if (Path.cwd()/'tokengate/demo.py').exists() else Path('/content/TokenGate')
if SOURCE == Path('/content/TokenGate'):
    if not SOURCE.exists():
        subprocess.run(['git','clone','-q','--branch','tokengate_demo',
                        'https://github.com/token-gate-llms/TokenGate',str(SOURCE)],check=True)
    else:
        subprocess.run(['git','-C',str(SOURCE),'fetch','origin','tokengate_demo'],check=True)
        subprocess.run(['git','-C',str(SOURCE),'switch','-C','tokengate_demo',
                        'origin/tokengate_demo'],check=True)
assert (SOURCE/'tokengate/demo.py').exists(), 'TokenGate demo helper is missing from this checkout.'
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(SOURCE),
                'transformers==5.5.0','matplotlib'],check=True)
sys.path.insert(0,str(SOURCE))
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
from tokengate import load_model, TokenGateConfig, TokenGateTrainer
from tokengate.demo import (niah_examples, niah_prompt, plain_results, gate_results,
                            result_figure, show_kept_tokens, release_niah_rows,
                            release_niah_config_kwargs)
print('GPU:', torch.cuda.get_device_name(0))

## 2. Inference

### Ordinary model: read the whole document

The unmodified base model answers the same three NIAH questions first. Its time to first token is the reference for the TokenGate results below.

In [ ]:
#@title Make three held-out NIAH prompts { display-mode: "form" }
samples = niah_examples(SOURCE, DOC_TOKENS)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
pairs = [niah_prompt(tok, row) for row in samples]
gate_prompts, plain_prompts = map(list, zip(*pairs))
for row in samples:
    print(f"{row['city']:7s} needle at {row['depth_actual']:.0%} · {row['question']}")

In [ ]:
#@title Run the ordinary model { display-mode: "form" }
if GPU_RUNTIME == 'A100':
    assert torch.cuda.get_device_properties(0).total_memory >= 35*2**30, 'Use an A100-class GPU for single-pass prefill.'
dtype = torch.float16 if GPU_RUNTIME == 'T4' else torch.bfloat16
PLAIN_CHUNK_TOKENS = 1024 if GPU_RUNTIME == 'T4' and MODEL == 'Qwen3-1.7B' and DOC_TOKENS > 4096 else None
print('Ordinary-model prefill:', f'cached {PLAIN_CHUNK_TOKENS}-token chunks' if PLAIN_CHUNK_TOKENS else 'single pass')
plain_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype,
    attn_implementation='sdpa').cuda().eval()
plain = plain_results(plain_model, tok, plain_prompts, samples,
    chunk_tokens=PLAIN_CHUNK_TOKENS)
for row in plain: print(f"{row['city']:7s} correct={bool(row['correct'])}  TTFT={row['prefill_s']:.2f}s")
del plain_model; gc.collect(); torch.cuda.empty_cache()

### Accelerate with TokenGate

`load_model` loads the base model, gate, and paired LoRA. `generate` uses the same prompts; `max_kept_prefill_tokens=K` controls how many document tokens survive prefill. The all-token arm shows the effect of the paired adapter without pruning.

In [ ]:
#@title Run TokenGate at each token budget { display-mode: "form" }
repo_id, revision = CHECKPOINTS[MODEL]
checkpoint = snapshot_download(repo_id, revision=revision)
gate = load_model(checkpoint, gate_attn='sdpa' if GPU_RUNTIME == 'T4' else 'flex', dtype=dtype)
gated = gate_results(gate, tok, gate_prompts, samples, BUDGETS)
for row in gated:
    print(f"{row['city']:7s} K={str(row['budget']):5s} correct={bool(row['correct'])} "
          f"TTFT={row['prefill_s']:.2f}s retained={row['retained']:.1%}")

### Interpret the results

The bars compare **time to first token** on the same three prompts and GPU; labels also show retrieval accuracy. On a free T4, Qwen3 uses cached 1,024-token chunks at 8k and 16k because its longer single-pass prefill can run out of memory. Qwen3.5 uses single-pass prefill through 8k; its 16k baseline ran out of memory on a T4. The paper uses a single-pass baseline, so **Qwen3's 8k/16k T4 speedups are not directly comparable to the paper's**. A100 uses single-pass prefill at every length. Three examples show the mechanism, not a reliable accuracy estimate.

In [ ]:
#@title Accuracy and speedup { display-mode: "form" }
from IPython.display import display
display(result_figure(plain, gated,
    baseline_name='chunked base (T4)' if PLAIN_CHUNK_TOKENS else f'single-pass base ({GPU_RUNTIME})'))

In [ ]:
#@title Which document tokens did the gate keep? { display-mode: "form" }
show_kept_tokens(gate, tok, gate_prompts[0], samples[0], budget=min(BUDGETS))

## 3. Optional: train and compare your own models

This runs the **stated NIAH-only checkpoint recipe** for the selected model: 12,000 synthetic training rows, 50 validation rows, gate plus LoRA, and a separate LoRA-only baseline trained on the same rows. The 150 reserved test rows are excluded from training and model selection. The gate stops at step 3,000 (Qwen3) or 18,000 (Qwen3.5); the LoRA-only run picks the lowest-validation-loss checkpoint among steps up to 10,000 or 18,000 respectively. The paper does not publish the exact LoRA-baseline stopping checkpoint, and the released gate's training provenance is not fully verifiable, so this is a reproducible recipe rather than a byte-for-byte reconstruction.

Use an A100 runtime; **four A100s** match the paper's batch layout. One or two A100s use gradient accumulation for the same effective batch of four and take longer. This full training path can take hours and needs substantial storage. The trained-model comparison below uses ordinary Hugging Face SDPA inference; the paper's compiled baseline may have different timing. Leave training off for the inference demo above.


In [ ]:
#@title Train TokenGate and the matched LoRA baseline (A100 only) { display-mode: "form" }
RUN_TRAINING = False #@param {type:"boolean"}
if RUN_TRAINING:
    assert GPU_RUNTIME == 'A100', 'Choose A100 in Demo settings and switch to an A100 runtime.'
    n_gpus = torch.cuda.device_count()
    assert n_gpus in (1, 2, 4) and all('A100' in torch.cuda.get_device_name(i) for i in range(n_gpus)), 'Use 1, 2, or 4 visible A100s.'
    del gate
    gc.collect(); torch.cuda.empty_cache()
    work = Path('/content/tokengate-training') if Path('/content').is_dir() else Path.cwd()/'tokengate-training'
    work = work/MODEL
    data_root = work/'data'
    train_rows, eval_rows = release_niah_rows(SOURCE, work)
    config = TokenGateConfig(**release_niah_config_kwargs(SOURCE, BASE_MODEL, work/'gate', data_root, n_gpus))
    trainer = TokenGateTrainer(config, train_dataset=train_rows, eval_dataset=eval_rows, task='niah_kamradt')
    gate_checkpoint = Path(config.output_dir)/'gate_final.pt'
    if not gate_checkpoint.exists():
        if MODEL == 'Qwen3-1.7B':
            from kernels.fa2bias.keybias import available as fa2_available
            if not fa2_available():
                subprocess.run([sys.executable, str(SOURCE/'kernels/fa2bias/build.py')], check=True)
        trainer.train()
    del trainer, train_rows, eval_rows
    gc.collect()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft>=0.15'], check=True)
    baseline_dir = work/'lora_baseline'
    if not (baseline_dir/'adapter_model.safetensors').exists():
        baseline_steps = 10000 if MODEL == 'Qwen3-1.7B' else 18000
        subprocess.run([sys.executable, '-m', 'torch.distributed.run',
                        f'--nproc_per_node={n_gpus}', '--master_port=29501',
                        '-m', 'tokengate.demo_baseline', '--base-model', BASE_MODEL,
                        '--data-root', str(data_root), '--eval-split', str(config.eval_split),
                        '--output-dir', str(baseline_dir), '--max-steps', str(baseline_steps),
                        '--grad-accum', str(4//n_gpus)], check=True)
    print('Gate:', gate_checkpoint, '| LoRA baseline:', baseline_dir)
else:
    print('Optional full training skipped; inference above used the released checkpoint.')


In [ ]:
#@title Compare your trained gate with its LoRA baseline (A100 only) { display-mode: "form" }
if RUN_TRAINING:
    from peft import PeftModel
    benchmark_rows = niah_examples(SOURCE, 16384)
    pairs = [niah_prompt(tok, row) for row in benchmark_rows]
    trained_gate_prompts, baseline_prompts = map(list, zip(*pairs))
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, attn_implementation='sdpa')
    tuned_base = PeftModel.from_pretrained(base, str(baseline_dir)).merge_and_unload().cuda().eval()
    tuned_plain = plain_results(tuned_base, tok, baseline_prompts, benchmark_rows)
    print('LoRA baseline:', sum(r['correct'] for r in tuned_plain), '/', len(tuned_plain), 'correct')
    del tuned_base, base
    gc.collect(); torch.cuda.empty_cache()
    own_gate = load_model(str(gate_checkpoint), gate_attn='fa2' if MODEL == 'Qwen3-1.7B' else 'flex', dtype=torch.bfloat16)
    own = gate_results(own_gate, tok, trained_gate_prompts, benchmark_rows, BUDGETS)
    for k in ('full',) + BUDGETS:
        rows = [r for r in own if r['budget'] == k]
        print(f'K={k}: {sum(r["correct"] for r in rows)}/{len(rows)} correct')
    display(result_figure(tuned_plain, own, baseline_name='LoRA-tuned base (SDPA, A100)'))
else:
    print('Run the optional training cell first.')
